# OpenPlaque — Image-Driven Coronary Tracking V2

This revision fixes the failure mode seen in the first image-driven tracker: very long cyclic chamber-boundary paths (roughly 800–1200 mm) could outrank true coronary-scale paths.

V2 uses **open-path topology**, a hard **20–220 mm path-length range**, sinuosity limits, small-structure image evidence, and plaque consistency as an independent selection/QC signal. It does **not** change canonical TPV.

Every persistent cache has a Boolean reuse flag near the top. All default to `True`:
- `True` + valid cache → reuse
- `True` + missing/invalid cache → recompute and cache
- `False` → force recomputation and update cache

CPR plaque views and source-volume PCAT remain separate coordinate systems. Research use only.


## Step 1 — Mount Google Drive

In [ ]:
# FIRST EXECUTABLE CELL — Drive mount must remain first.
from google.colab import drive
drive.mount('/content/drive')


## Step 2 — Cache / reuse controls

In [ ]:
# Default: reuse every valid cache.
REUSE_SERIES_SELECTION = True
REUSE_PLAQUE_MASKS = True
REUSE_TRACKING = True
REUSE_CANDIDATE_FIGURE = True
REUSE_ROADMAPS = True
REUSE_PCAT_FIGURES = True
REUSE_DASHBOARD = True
REUSE_REPORT_PACKAGE = True


## Step 3 — Install this V2 branch

In [ ]:
!rm -rf /content/OpenPlaque
!git clone -q --branch image-driven-tracking-v2-from-main --single-branch https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
!pip -q install -r /content/OpenPlaque/requirements-colab.txt

import sys
sys.path.insert(0, '/content/OpenPlaque/src')
import pandas as pd
from IPython.display import display, Image
from openplaque.tracking_v2_workflow import TrackingV2Workflow

print('OpenPlaque V2 tracker ready.')


## Step 4 — Initialize and inspect the cache plan

This table tells you, *before expensive work starts*, whether each component will be reused or recomputed.


In [ ]:
REUSE = {
    'series_selection': REUSE_SERIES_SELECTION,
    'plaque_masks': REUSE_PLAQUE_MASKS,
    'tracking': REUSE_TRACKING,
    'candidate_figure': REUSE_CANDIDATE_FIGURE,
    'roadmaps': REUSE_ROADMAPS,
    'pcat_figures': REUSE_PCAT_FIGURES,
    'dashboard': REUSE_DASHBOARD,
    'report_package': REUSE_REPORT_PACKAGE,
}
wf = TrackingV2Workflow('/content/drive/MyDrive/OpenPlaque', REUSE)
display(wf.cache_status())


## Step 5 — Prepare DICOM and coronary series selection

In [ ]:
series_map = wf.prepare_inputs()
print('Series:', series_map)


## Step 6 — Load canonical plaque masks

V2 first looks in its own normalized cache, then the previous cache-controlled run, then established legacy segmentation caches. It recomputes with nnU-Net only when reuse is disabled or no valid plaque cache is available.


In [ ]:
cache_qc = wf.load_plaque_masks()
display(cache_qc)


## Step 7 — V2 open-path coronary tracking

Hard rejection rules prevent the prior chamber-loop failure:
- path must be open / acyclic,
- length must be 20–220 mm,
- sinuosity must be ≤2.6,
- path HU must be plausible,
- if canonical plaque exists, zero plaque capture is a QC failure.

Plaque does not generate the path; it is used as a selection/QC consistency signal.


In [ ]:
tracking_qc = wf.track_coronaries()
display(tracking_qc)


## Step 8 — Review the top three V2 candidates for each vessel

In [ ]:
candidate_png = wf.plot_candidates()
print('Saved:', candidate_png)
display(Image(filename=str(candidate_png)))


## Step 9 — Create straightened plaque roadmaps

These are visualization-only. A panel explicitly carries the V2 tracking QC status.


In [ ]:
roadmap_png = wf.plot_roadmaps()
print('Saved:', roadmap_png)
display(Image(filename=str(roadmap_png)))
if wf.along_df is not None:
    display(wf.along_df.head(40))


## Step 10 — RCA PCAT figures

The default is to reuse a validated cached PCAT visualization. Set `REUSE_PCAT_FIGURES=False` at the top to regenerate it from the frozen RCA centerline/radius/aorta inputs.


In [ ]:
pcat_files = wf.pcat_figures()
print(pcat_files)
display(Image(filename=str(pcat_files['cross_sections'])))
display(Image(filename=str(pcat_files['ribbon'])))


## Step 11 — Build the summary dashboard

In [ ]:
dashboard = wf.plot_dashboard()
print('Saved:', dashboard)
display(Image(filename=str(dashboard)))


## Step 12 — Package the report-back ZIP

`cache_provenance.csv` is included in the ZIP in this revision.


In [ ]:
zip_path = wf.package_report()
print('Report ZIP:', zip_path)
print('Drive search: https://drive.google.com/drive/u/0/search?q=OPENPLAQUE_IMAGE_DRIVEN_TRACKING_V2_REPORT_BACK.zip')
print('\nCache provenance:')
display(pd.read_csv(wf.out / 'cache_provenance.csv'))
